# ☕ Welcome to Sunny Bay Roastery

## 🎯 Objectives
By the end of this lab, you will:
- Have a **Databricks** workspace (Free Edition works too).
- Have run **one notebook — this Lab 0** — that deploys the whole workshop: your **catalog and schemas** (`bronze`, `silver`, `gold`), the Sunny Bay sales data, the medallion pipeline, the metric view, the pre-built **Sales Genie**, and the **dashboards**.
- Understand the **Sunny Bay Roastery** story and your role in it.

## 🚀 Set Up Your Workspace (One Click)

Setting up is one click: **click `Run all` at the top of this notebook** (serverless — no cluster to pick). Lab 0 creates your catalog, deploys the workshop assets, and runs the setup job end-to-end. It takes about **10–15 minutes**; when the last cell prints **✅ SETUP COMPLETE**, your workspace is ready and you can head to **Lab 1**.

Before you run it, you can optionally change the two parameters below — the defaults work as-is.

In [ ]:
# ── Edit these if you need to (the defaults work as-is) ──────────────────
catalog = "sunny_bay_roastery"   # the Unity Catalog catalog to build everything in

# Optional. Set this only if several people share the SAME catalog and schema
# (for example "alice"). It is added to every object name — tables, views and
# the metric view — so your objects don't collide with a colleague's. Leave it
# empty ("") if you are working on your own.
prefix = ""
# ─────────────────────────────────────────────────────────────────────────

# Schema names are fixed by the workshop pipeline — leave these as-is.
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

> [!NOTE]
> On some Databricks workspaces you may not be able to create a new catalog. If so, set `catalog` above to the name of a catalog you already own, and the setup will use it automatically.

In [ ]:
import re
from pathlib import Path

# Normalize the prefix: add a trailing underscore so names read cleanly
# (e.g. "alice" -> "alice_dim_customer"). An empty prefix leaves names unchanged.
prefix = prefix.strip()
if prefix and not prefix.endswith("_"):
    prefix += "_"

params = {
    "catalog": catalog,
    "bronze_schema": bronze_schema,
    "silver_schema": silver_schema,
    "gold_schema": gold_schema,
    # Quoted so an empty prefix stays an empty string ("") instead of YAML null.
    "prefix": f'"{prefix}"',
}

path = Path("../bundle/databricks.yml")
text = path.read_text()
for key, value in params.items():
    text = re.sub(rf"^(      {key}: ).*$", lambda m, v=value: m.group(1) + v, text, flags=re.M)
path.write_text(text)

try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    print(f"✅ Catalog `{catalog}` is ready.")
except Exception:
    # Some workspaces restrict catalog creation. Fall back to an existing
    # catalog — make sure the `catalog` value above matches a catalog you own.
    spark.sql(f"USE CATALOG {catalog}")
    print(f"⚠️ Could not create catalog `{catalog}` (creation may be restricted "
          f"on this workspace). Using the existing catalog `{catalog}` instead.")

In [ ]:
from pathlib import Path

path = Path("../bundle/src/dashboards/dashboard_final.lvdash.json")
text = path.read_text()
text = (
    text.replace('__CATALOG__', catalog)
        .replace('__GOLD_SCHEMA__', gold_schema)
        .replace('__PREFIX__', prefix)
)
path.write_text(text)

### Deploy and run (automatic)

The cells below install the Databricks CLI, deploy the workshop assets (the setup job, the Lakeflow medallion pipeline, and the two AI/BI dashboards) using your own workspace credentials, and then run the setup job to generate the data, build the medallion tables and metric view, and pre-build the Sales Genie. This is the longest part of the run.

In [ ]:
import os
import re
import subprocess
import tempfile

# Lab 0 lives in <repo>/labs, so the bundle root (where databricks.yml lives) is one
# directory up, in <repo>/bundle.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
bundle_root = os.path.abspath("../bundle")
if not os.path.exists(os.path.join(bundle_root, "databricks.yml")):
    raise RuntimeError(
        f"databricks.yml not found at {bundle_root}. Run this notebook from inside the "
        "cloned Git Folder (labs/Lab 0 - Intro), next to the bundle/ tree."
    )

# Install the Databricks CLI. The setup script picks its own bin dir and prints
# "Installed Databricks CLI vX.Y.Z at <path>." — parse that path.
_install_dir = tempfile.mkdtemp(prefix="dbcli_")
_p = subprocess.run(
    ["bash", "-c",
     f"curl -fsSL -m 90 https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh "
     f"| sh -s -- {_install_dir}"],
    capture_output=True, text=True,
)
_m = re.search(r"Installed Databricks CLI \S+ at (\S+?)\.?$", _p.stdout.strip(), re.M)
if _m and os.path.exists(_m.group(1)):
    CLI = _m.group(1)
elif os.path.exists(os.path.join(_install_dir, "databricks")):
    CLI = os.path.join(_install_dir, "databricks")
else:
    _which = subprocess.run(["bash", "-c", "command -v databricks || true"],
                            capture_output=True, text=True).stdout.strip()
    CLI = _which if _which and os.path.exists(_which) else None
if not CLI:
    raise RuntimeError(f"Could not install the Databricks CLI.\nstdout={_p.stdout}\nstderr={_p.stderr}")

# Authenticate with the notebook's own token so deploy/run act as you. Pin the bundle
# root so the CLI finds databricks.yml when we run from the Git Folder.
cli_env = dict(
    os.environ,
    DATABRICKS_HOST=ctx.apiUrl().get(),
    DATABRICKS_TOKEN=ctx.apiToken().get(),
    DATABRICKS_BUNDLE_ROOT=bundle_root,
)


def run_cli(args):
    """Run the CLI streaming combined output into the notebook, raise on failure."""
    print(f"$ databricks {' '.join(args)}")
    proc = subprocess.Popen(
        [CLI, *args], cwd=bundle_root, env=cli_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"`databricks {' '.join(args)}` failed (exit {proc.returncode}).")


print(subprocess.run([CLI, "--version"], capture_output=True, text=True).stdout.strip())
run_cli(["current-user", "me", "-o", "json"])

In [ ]:
# Deploy the bundle: the Sunny Bay Roastery Job, the medallion Lakeflow pipeline, and the
# two AI/BI dashboards — all pointed at the catalog/prefix you set above (already written
# into databricks.yml a few cells up).
run_cli(["bundle", "deploy", "-t", "dev", "--force-lock"])
print("✅ Bundle deployed (job, pipeline, and dashboards).")

In [ ]:
# Run the setup job end-to-end (~10–15 min): generates the sales data, runs the
# bronze→silver→gold pipeline, builds the metric view, and pre-builds the Sales Genie.
run_cli(["bundle", "run", "sunny_bay_roastery_job", "-t", "dev", "--restart"])

In [ ]:
p = prefix or ""
print("=" * 65)
print("✅  SETUP COMPLETE  —  you are ready for Dashboard in a Day!")
print("=" * 65)
print()
print(f"Everything is in catalog `{catalog}`:")
print(f"  🥉 Bronze raw    : {catalog}.{bronze_schema}.raw (Volume) + generated CSV/Parquet")
print(f"  🥈 Silver / 🥇 Gold star schema : {catalog}.{gold_schema}.{p}fact_coffee_sales + {p}dim_*")
print(f"  📐 Metric view   : {catalog}.{gold_schema}.{p}sm_fact_coffee_sales_genie")
print(f"  🧞 Sales Genie   : pre-built over the metric view")
print(f'  📊 Dashboards    : "[Template]" and "[Final]" Sunny Bay Roastery - Sales Report')
print()
print("Next: open  labs/Lab 1  and follow along!")

## ☕ Story Setup

In 2013, the very same year **Databricks** was founded, a small team of coffee enthusiasts opened a café in **San Francisco**. They called it **Sunny Bay Roastery**.

At first, they were just another boutique coffee shop, but their obsession with precision, data, and quality soon made them a local favorite. Every espresso shot was logged.

Over the next decade, Sunny Bay grew to five stores across the Bay Area. Then, in 2020, when the pandemic hit, foot traffic dropped overnight. The company had to act fast.

Online coffee bean sales exploded as people became **home baristas**, experimenting with grinders and brewing ratios while stuck at home.

Today, in 2025, Sunny Bay Roastery stands at a crossroads.  
Its CEO, **Mr. Bean**, wants to understand:
> "What really drives our coffee sales?  
>  How do seasons, holidays, and online vs. in-store trends shape our future?"

Unfortunately, the company's data is scattered:
- Some comes from old **in-store point-of-sale systems**.  
- Some from **e-commerce logs**.  
- And some information are in **Excel files** sitting in Mr. Bean's inbox.

You've just been hired as the company's first **Head of Data & Analytics**.  
Your mission: **build a unified data platform** that can turn these fragments into insight — and prepare Sunny Bay Roastery for its next phase of growth.